# Remapping Metrics

This notebook quantifies the degree of **place cell remapping** between LM8 and LM8_R45 using the LM8-trained place cell ensemble.

The place cells are activated by observations collected in each environment. The resulting activation matrices form the basis for computing spatial and population-level remapping metrics in subsequent cells.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import xml.etree.ElementTree as ET
os.chdir('..')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Force white background regardless of PyCharm's dark theme
plt.style.use('default')
mpl.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'savefig.facecolor': 'white',
    'text.color'       : 'black',
    'axes.labelcolor'  : 'black',
    'xtick.color'      : 'black',
    'ytick.color'      : 'black',
    'axes.edgecolor'   : 'black',
})

from realm_tools.experiment_lib.loggers import PovDataset, PlaceCellEnsemble
from realm_tools.place_cell_lib import VisualPlaceCellEnsemble
from realm_tools.simulation_lib.environment_parser import parse_all_walls
from realm_tools.simulation_lib.start_position_generator import _outer_boundary
from realm_tools.image_lib.analysis_plots import _draw_maze, _layout

---
## Configuration

In [ ]:
TRAIN_MAZE           = 'lm8'
TEST_MAZE            = 'lm8_r45'
CENTROID_THRESHOLD   = 0.4
STABILITY_SEED       = 42     # fixed seed for the within-environment half-split

PLACE_CELL_PATH = f'data/vpce/place_cells/{TRAIN_MAZE}'
TRAIN_DATA_PATH = f'data/vpce/collect_data/{TRAIN_MAZE}'
TEST_DATA_PATH  = f'data/vpce/collect_data/{TEST_MAZE}'
TRAIN_MAZE_XML  = f'simulation/worlds/environments/vpce/{TRAIN_MAZE}.xml'
BASE_IMG_PATH   = f'analysis/base_figures/{TRAIN_MAZE}.png'
FLIP_BASE_IMG   = True

SAVE_DIR = 'analysis/figures/remapping'
os.makedirs(SAVE_DIR, exist_ok=True)

def save(filename):
    path = os.path.join(SAVE_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"Saved → {path}")

In [ ]:
pc_model = PlaceCellEnsemble.load(PLACE_CELL_PATH)
ensemble = VisualPlaceCellEnsemble(pc_model.centers, pc_model.radii)

print(f"Place cells : {ensemble.n_cells}")
print(f"Feature dim : {ensemble.feature_dim}")
print(f"Method      : {pc_model.method}")
print(f"Trained on  : {pc_model.maze}")

---
## Compute Activations

The same place cell ensemble is activated by observations from both environments. The resulting matrices `A_lm8` and `A_lm8_r45` have shape `(N_observations, N_cells)` and are the input to all downstream metrics.

In [ ]:
# LM8 — baseline environment
train_ds       = PovDataset.load_dataset(TRAIN_DATA_PATH)
features_lm8   = np.array(train_ds.multimodal_features)
poses_lm8      = np.stack([train_ds.x, train_ds.y, train_ds.theta], axis=1)
A_lm8          = ensemble.activate(features_lm8)

# LM8_R45 — landmarks rotated 45°
test_ds        = PovDataset.load_dataset(TEST_DATA_PATH)
features_r45   = np.array(test_ds.multimodal_features)
poses_r45      = np.stack([test_ds.x, test_ds.y, test_ds.theta], axis=1)
A_r45          = ensemble.activate(features_r45)

print(f"A_lm8  : {A_lm8.shape}  —  {TRAIN_MAZE.upper()}")
print(f"A_r45  : {A_r45.shape}  —  {TEST_MAZE.upper()}")

---
## Peak Activation Location

For each place cell the **activation-weighted centroid** of all observation positions gives the spatial location where that cell's activation is concentrated:

$$\bar{x}_i = \frac{\sum_n a_i(n)\, x_n}{\sum_n a_i(n)}, \qquad \bar{y}_i = \frac{\sum_n a_i(n)\, y_n}{\sum_n a_i(n)}$$

This is more robust than argmax — it reflects the full distribution of activation across the maze rather than the single highest-activation observation.

In [ ]:
def weighted_centroids(activations, poses, threshold=0.0):
    """
    Compute the activation-weighted centroid (x, y) for each place cell.

    Activations below `threshold * cell_max` are zeroed before computing
    the weighted mean, ignoring weakly activated locations.

    Parameters
    ----------
    activations : np.ndarray, shape (N, K)
    poses       : np.ndarray, shape (N, 3)  — columns: x, y, theta
    threshold   : float  — fraction of each cell's max activation below which
                           observations are ignored (default 0 = use all)

    Returns
    -------
    np.ndarray, shape (K, 2)  — (x, y) centroid per cell
    """
    xy      = poses[:, :2]
    weights = activations.copy()

    if threshold > 0:
        cell_max = weights.max(axis=0, keepdims=True)   # (1, K)
        weights[weights < threshold * cell_max] = 0

    totals = weights.sum(axis=0, keepdims=True).T        # (K, 1)
    return (weights.T @ xy) / totals                     # (K, 2)


centroids_lm8 = weighted_centroids(A_lm8, poses_lm8, threshold=CENTROID_THRESHOLD)
centroids_r45 = weighted_centroids(A_r45, poses_r45, threshold=CENTROID_THRESHOLD)

print(f"Threshold        : {CENTROID_THRESHOLD:.0%} of each cell's max activation")
print(f"Centroids LM8    : {centroids_lm8.shape}")
print(f"Centroids R45    : {centroids_r45.shape}")

---
## Centroid Shift Map

Each place cell is represented by two dots superimposed on the LM8 base figure:

- 🟢 **Green** — activation-weighted centroid in LM8 (baseline)
- 🔴 **Red** — activation-weighted centroid in LM8_R45 (rotated landmarks)

A line connects each pair to make the shift direction visible. Cells whose centroids shift substantially between environments are remapping in response to the landmark rotation.

In [ ]:
# --- Base image extent from maze XML ---
walls    = parse_all_walls(ET.parse(TRAIN_MAZE_XML).getroot())
boundary = _outer_boundary(walls)
bminx, bminy, bmaxx, bmaxy = boundary.bounds
base_extent = [bminx, bmaxx, bminy, bmaxy]

base_img = plt.imread(BASE_IMG_PATH)
if FLIP_BASE_IMG:
    base_img = np.flipud(base_img)

n_cells = len(centroids_lm8)
ncols   = int(np.ceil(np.sqrt(n_cells)))
nrows   = int(np.ceil(n_cells / ncols))

fig_w, fig_h, subplot_top, cbar_bot, cbar_h_n, title_y = _layout(nrows, ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), facecolor='white')
axes = axes.flatten()

for i in range(n_cells):
    ax = axes[i]
    ax.set_facecolor('white')
    ax.imshow(base_img, extent=base_extent, origin='lower', aspect='equal', zorder=0)
    _draw_maze(ax, TRAIN_MAZE_XML)
    ax.plot([centroids_lm8[i, 0], centroids_r45[i, 0]],
            [centroids_lm8[i, 1], centroids_r45[i, 1]],
            color='grey', linewidth=1.2, alpha=0.6, zorder=1)
    ax.scatter(centroids_lm8[i, 0], centroids_lm8[i, 1],
               c='green', s=60, zorder=3, edgecolors='darkgreen', linewidths=0.6)
    ax.scatter(centroids_r45[i, 0], centroids_r45[i, 1],
               c='red', s=60, zorder=3, edgecolors='darkred', linewidths=0.6)
    ax.set_xlim(base_extent[0], base_extent[1])
    ax.set_ylim(base_extent[2], base_extent[3])
    ax.set_title(f'Cell {i}', fontsize=39, color='black', pad=4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

for j in range(n_cells, len(axes)):
    axes[j].set_visible(False)

plt.subplots_adjust(top=subplot_top, bottom=0.01, hspace=0.35, wspace=0.2)
fig.suptitle(
    f'Place Cell Centroid Shift — {TRAIN_MAZE.upper()} vs {TEST_MAZE.upper()}\n'
    f'{pc_model.method.upper()}  K={n_cells}',
    fontsize=54, color='black', y=title_y
)

legend_ax = fig.add_axes([0.15, cbar_bot, 0.7, cbar_h_n * 3])
legend_ax.axis('off')
legend_ax.legend(
    handles=[
        mpatches.Patch(color='green', label=f'{TRAIN_MAZE.upper()} — baseline'),
        mpatches.Patch(color='red',   label=f'{TEST_MAZE.upper()} — landmarks +45°'),
    ],
    loc='center', ncol=2, fontsize=36, framealpha=0.0
)

save(f'centroid_shift_map_{TRAIN_MAZE}_{TEST_MAZE}.png')
plt.show()

---
## Metric 1 — Angular Shift Distribution

For each place cell the polar angle of its peak activation centroid is computed relative to the environment centre, and the difference between environments is taken as the angular shift:

$$\theta^{c}_{\text{env}} = \text{atan2}(y - c_y,\; x - c_x)$$

$$\Delta\theta^{c} = \left[(\theta^{c}_{\text{R45}} - \theta^{c}_{\text{LM8}} + 180) \;\bmod\; 360\right] - 180$$

The modular arithmetic wraps $\Delta\theta^{c}$ to $[-180°, 180°]$. Under landmark-controlled remapping the expected value is $\Delta\theta^{c} \approx -45°$ (clockwise). We report the mean $\pm$ standard deviation and the fraction of cells satisfying $-60° \leq \Delta\theta^{c} \leq -30°$.

In [ ]:
cx, cy = 0.0, 0.0

theta_lm8 = np.degrees(np.arctan2(centroids_lm8[:, 1] - cy, centroids_lm8[:, 0] - cx))
theta_r45 = np.degrees(np.arctan2(centroids_r45[:, 1] - cy, centroids_r45[:, 0] - cx))
delta_theta = ((theta_r45 - theta_lm8 + 180) % 360) - 180

mean_shift = delta_theta.mean()
std_shift  = delta_theta.std()
in_range   = (delta_theta >= -60) & (delta_theta <= -30)
frac_range = in_range.mean()

print(f"Mean shift           : {mean_shift:.2f}°")
print(f"Std deviation        : {std_shift:.2f}°")
print(f"Expected (landmark)  : -45.00°")
print(f"Cells in [-60°,-30°] : {in_range.sum()} / {len(delta_theta)}  ({frac_range:.1%})")

fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor('white')
ax.hist(delta_theta, bins=36, range=(-180, 180),
        edgecolor='black', color='steelblue', alpha=0.85, label='Place cells')
ax.axvline(-45, color='red', linestyle='--', linewidth=2, label='Expected  −45°')
ax.axvspan(-60, -30, alpha=0.12, color='red', label='Acceptance band  [−60°, −30°]')
ax.axvline(mean_shift, color='black', linestyle='-', linewidth=1.8,
           label=f'Mean  {mean_shift:.1f}°')
ax.set_xlabel('Angular shift Δθ  (degrees)', fontsize=13, color='black')
ax.set_ylabel('Number of cells', fontsize=13, color='black')
ax.set_title(
    f'Angular Shift Distribution — {TRAIN_MAZE.upper()} → {TEST_MAZE.upper()}\n'
    f'mean = {mean_shift:.1f}°,  σ = {std_shift:.1f}°,  '
    f'{frac_range:.1%} of cells in [−60°, −30°]',
    fontsize=13, color='black'
)
ax.set_xlim(-180, 180)
ax.set_xticks(range(-180, 181, 45))
ax.tick_params(colors='black')
ax.legend(fontsize=11)
plt.tight_layout()
save(f'angular_shift_{TRAIN_MAZE}_{TEST_MAZE}.png')
plt.show()

---
## Metric 2 — Radial Displacement

A field may shift angularly but also migrate to a different distance from the centre, which would suggest the model found an unrelated cluster rather than tracking the landmark. Radial displacement checks that fields remain at approximately the same eccentricity:

$$r^{c}_{\text{env}} = \left\|\mathbf{p}^{c}_{\text{env}} - \mathbf{o}\right\|_2$$

$$\delta r^{c} = \left| r^{c}_{\text{R45}} - r^{c}_{\text{LM8}} \right|$$

A small mean $\overline{\delta r}$ relative to the environment radius confirms that fields rotate without migrating inward or outward.

In [ ]:
origin = np.array([cx, cy])

r_lm8   = np.linalg.norm(centroids_lm8 - origin, axis=1)
r_r45   = np.linalg.norm(centroids_r45 - origin, axis=1)
delta_r = np.abs(r_r45 - r_lm8)

env_radius       = (boundary.bounds[2] - boundary.bounds[0]) / 2
mean_delta_r     = delta_r.mean()
median_delta_r   = np.median(delta_r)
rel_displacement = mean_delta_r / env_radius

print(f"Environment radius       : {env_radius:.3f} m")
print(f"Mean δr                  : {mean_delta_r:.4f} m")
print(f"Median δr                : {median_delta_r:.4f} m")
print(f"Mean δr / env radius     : {rel_displacement:.3f}  ({rel_displacement:.1%})")

fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor('white')
ax.hist(delta_r, bins=40, edgecolor='black', color='steelblue', alpha=0.85)
ax.axvline(mean_delta_r,   color='black', linestyle='-',  linewidth=2,
           label=f'Mean  {mean_delta_r:.3f} m')
ax.axvline(median_delta_r, color='black', linestyle='--', linewidth=1.5,
           label=f'Median  {median_delta_r:.3f} m')
ax.set_xlabel('Radial displacement δr  (m)', fontsize=13, color='black')
ax.set_ylabel('Number of cells',             fontsize=13, color='black')
ax.set_title(
    f'Radial Displacement Distribution — {TRAIN_MAZE.upper()} → {TEST_MAZE.upper()}\n'
    f'mean δr = {mean_delta_r:.3f} m  ({rel_displacement:.1%} of env radius)',
    fontsize=13, color='black'
)
ax.tick_params(colors='black')
ax.legend(fontsize=11)
plt.tight_layout()
save(f'radial_displacement_{TRAIN_MAZE}_{TEST_MAZE}.png')
plt.show()

---
## Metric 3 — Population Vector (PV) Overlap

PV overlap is the standard biological remapping metric. At each spatial location the cosine similarity between the population activation vectors from the two environments is computed:

$$\text{PV}(\mathbf{x}) = \frac{\mathbf{a}_{\text{LM8}}(\mathbf{x}) \cdot \mathbf{a}_{\text{R45}}(\mathbf{x})}{\left\|\mathbf{a}_{\text{LM8}}(\mathbf{x})\right\|_2 \left\|\mathbf{a}_{\text{R45}}(\mathbf{x})\right\|_2}$$

A value of 1 means the population code is identical; 0 means the representations are orthogonal (complete remapping). PV overlap is reported as a function of distance from the environment centre, averaged across observations within each distance bin.

In [ ]:
dot        = np.sum(A_lm8 * A_r45, axis=1)
norm_lm8   = np.linalg.norm(A_lm8, axis=1)
norm_r45   = np.linalg.norm(A_r45, axis=1)
pv_overlap = dot / (norm_lm8 * norm_r45 + 1e-12)

distances = np.linalg.norm(poses_lm8[:, :2] - origin, axis=1)

print(f"Mean PV overlap : {pv_overlap.mean():.4f}")
print(f"Std PV overlap  : {pv_overlap.std():.4f}")
print(f"Min / Max       : {pv_overlap.min():.4f} / {pv_overlap.max():.4f}")

N_BINS      = 20
bin_edges   = np.linspace(0, distances.max(), N_BINS + 1)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_mean    = np.full(N_BINS, np.nan)
bin_sem     = np.full(N_BINS, np.nan)

for k in range(N_BINS):
    mask = (distances >= bin_edges[k]) & (distances < bin_edges[k + 1])
    if mask.sum() > 1:
        vals         = pv_overlap[mask]
        bin_mean[k]  = vals.mean()
        bin_sem[k]   = vals.std() / np.sqrt(mask.sum())

fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor('white')
ax.scatter(distances, pv_overlap, s=3, alpha=0.15, color='steelblue', zorder=1)
valid = ~np.isnan(bin_mean)
ax.errorbar(bin_centres[valid], bin_mean[valid], yerr=bin_sem[valid],
            fmt='o-', color='black', linewidth=2, markersize=5,
            capsize=3, label='Mean ± SEM per bin', zorder=3)
ax.axhline(1.0, color='green',  linestyle=':', linewidth=1.2, label='PV = 1  (identical)')
ax.axhline(0.0, color='orange', linestyle=':', linewidth=1.2, label='PV = 0  (orthogonal)')
ax.axvline(env_radius, color='red', linestyle='--', linewidth=1.2,
           label=f'Environment radius  {env_radius:.2f} m')
ax.set_xlabel('Distance from environment centre (m)', fontsize=13, color='black')
ax.set_ylabel('PV overlap (cosine similarity)',        fontsize=13, color='black')
ax.set_title(
    f'Population Vector Overlap vs Distance — {TRAIN_MAZE.upper()} vs {TEST_MAZE.upper()}\n'
    f'mean PV = {pv_overlap.mean():.3f}  ±  {pv_overlap.std():.3f}',
    fontsize=13, color='black'
)
ax.set_ylim(-0.05, 1.05)
ax.set_xlim(left=0)
ax.tick_params(colors='black')
ax.legend(fontsize=10)
plt.tight_layout()
save(f'pv_overlap_{TRAIN_MAZE}_{TEST_MAZE}.png')
plt.show()

---
## Metric 4 — Within-Environment Field Stability

Remapping can only be claimed if fields are stable within each environment independently. Instability within a single environment would indicate that the between-environment shifts are noise rather than signal.

Each environment's observation set is randomly split into two equal halves $\mathcal{H}_1$ and $\mathcal{H}_2$. Peak locations are computed independently on each half, and the Euclidean distance between them is taken as the stability score per cell:

$$\text{Stability}^{c} = \left\|\mathbf{p}^{c}_{\mathcal{H}_1} - \mathbf{p}^{c}_{\mathcal{H}_2}\right\|_2$$

Mean within-environment stability must be substantially smaller than the mean between-environment displacement for the remapping result to be interpretable.

In [ ]:
rng = np.random.default_rng(STABILITY_SEED)

def half_split_stability(A, poses, threshold, rng):
    N   = len(poses)
    idx = rng.permutation(N)
    h1, h2 = idx[:N // 2], idx[N // 2:]
    c1 = weighted_centroids(A[h1], poses[h1], threshold=threshold)
    c2 = weighted_centroids(A[h2], poses[h2], threshold=threshold)
    return np.linalg.norm(c1 - c2, axis=1)

stability_lm8 = half_split_stability(A_lm8, poses_lm8, CENTROID_THRESHOLD, rng)
stability_r45 = half_split_stability(A_r45, poses_r45, CENTROID_THRESHOLD, rng)
between_disp  = np.linalg.norm(centroids_lm8 - centroids_r45, axis=1)

print(f"Within-env stability  LM8   : {stability_lm8.mean():.4f} m  ± {stability_lm8.std():.4f}")
print(f"Within-env stability  R45   : {stability_r45.mean():.4f} m  ± {stability_r45.std():.4f}")
print(f"Between-env displacement    : {between_disp.mean():.4f} m  ± {between_disp.std():.4f}")
print(f"\nRatio (between / LM8 within): {between_disp.mean() / stability_lm8.mean():.2f}×")
print(f"Ratio (between / R45 within): {between_disp.mean() / stability_r45.mean():.2f}×")

fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor('white')

bins = np.linspace(0, max(stability_lm8.max(), stability_r45.max(),
                           between_disp.max()) * 1.05, 40)

ax.hist(stability_lm8, bins=bins, alpha=0.65, color='green',
        edgecolor='black', label=f'{TRAIN_MAZE.upper()} within-env  (mean {stability_lm8.mean():.3f} m)')
ax.hist(stability_r45, bins=bins, alpha=0.65, color='steelblue',
        edgecolor='black', label=f'{TEST_MAZE.upper()} within-env  (mean {stability_r45.mean():.3f} m)')
ax.axvline(between_disp.mean(), color='red', linestyle='--', linewidth=2,
           label=f'Between-env displacement  (mean {between_disp.mean():.3f} m)')

ax.set_xlabel('Field displacement  (m)', fontsize=13, color='black')
ax.set_ylabel('Number of cells',         fontsize=13, color='black')
ax.set_title(
    f'Within-Environment Field Stability — {TRAIN_MAZE.upper()} & {TEST_MAZE.upper()}\n'
    f'between-env / within-env ratio: '
    f'{between_disp.mean() / stability_lm8.mean():.1f}× (LM8)  '
    f'{between_disp.mean() / stability_r45.mean():.1f}× (R45)',
    fontsize=13, color='black'
)
ax.tick_params(colors='black')
ax.legend(fontsize=11)
plt.tight_layout()
save(f'field_stability_{TRAIN_MAZE}_{TEST_MAZE}.png')
plt.show()